In [ ]:
!pip install ultralytics opencv-python matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/Safe/archive_00.zip" -d "/content/drive/MyDrive/Safe/archive_00"

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [ ]:
# archive_00 압축 해제 경로
BASE_DIR = "/content/drive/MyDrive/Safe/archive_00"

# 처리할 split 목록
SPLITS = ["train", "val", "test"]

# 크롭 결과 저장 폴더
CROP_OUTPUT_DIR = "/content/drive/MyDrive/Safe/cropped_data_archive"

os.makedirs(os.path.join(CROP_OUTPUT_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(CROP_OUTPUT_DIR, "labels"), exist_ok=True)

설정값

In [ ]:
UPPER_BODY_RATIO = 0.65       # 전신 bbox에서 상체 비율 (상위 65%)
PERSON_CONFIDENCE = 0.25      # 사람 감지 신뢰도 임계값
MIN_CROP_SIZE = 50            # 크롭 최소 크기 (px)
LABEL_CONTAIN_RATIO = 0.5     # 라벨 bbox가 크롭 영역에 50% 이상 포함 시 매칭
IOU_DUPLICATE_THRESHOLD = 0.5 # 사람 bbox 중복 제거 IoU 임계값

# 클래스 이름 매핑 (시각화용)
CLASS_NAMES = {0: "helmet", 1: "vest", 2: "no-helmet", 3: "no-vest"}

print(f"베이스 경로: {BASE_DIR}")
print(f"처리 대상:   {SPLITS}")
print(f"크롭 저장:   {CROP_OUTPUT_DIR}")
print(f"상체 비율:   상위 {UPPER_BODY_RATIO * 100:.0f}%")
print(f"라벨 포함 기준: {LABEL_CONTAIN_RATIO * 100:.0f}%")
print()

베이스 경로: /content/drive/MyDrive/Safe/archive_00
처리 대상:   ['train', 'val', 'test']
크롭 저장:   /content/drive/MyDrive/Safe/cropped_data_archive
상체 비율:   상위 65%
라벨 포함 기준: 50%



모델 로드

In [ ]:
model = YOLO("yolo11m.pt")  # 자동 다운로드됨
print("모델 로드 완료\n")

모델 로드 완료



핵심함수

In [ ]:
def calculate_iou(box1, box2):
    """두 bbox의 IoU 계산 (x1, y1, x2, y2 형식)"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def remove_duplicate_boxes(persons, iou_threshold=0.5):
    """IoU가 높은 중복 bbox 제거"""
    if len(persons) <= 1:
        return persons

    keep = []
    used = set()
    sorted_persons = sorted(persons, key=lambda p: p['confidence'], reverse=True)

    for i, p in enumerate(sorted_persons):
        if i in used:
            continue
        keep.append(p)
        for j in range(i + 1, len(sorted_persons)):
            if j in used:
                continue
            if calculate_iou(p['bbox'], sorted_persons[j]['bbox']) > iou_threshold:
                used.add(j)
    return keep


def crop_upper_body(image, bbox):
    """사람 bbox에서 상체 영역만 크롭"""
    x1, y1, x2, y2 = bbox
    h = y2 - y1
    w = x2 - x1
    img_h, img_w = image.shape[:2]

    aspect_ratio = h / w if w > 0 else 1

    if aspect_ratio >= 2.0:
        # 전신: 상위 65%만
        crop_y2 = int(y1 + h * UPPER_BODY_RATIO)
    elif aspect_ratio >= 1.2:
        # 중간: 아래로 약간 확장
        crop_y2 = min(int(y2 + h * 0.1), img_h)
    else:
        # 이미 상체 위주: 전체 사용
        crop_y2 = min(int(y2 + h * 0.05), img_h)

    crop_x1 = max(0, x1)
    crop_y1 = max(0, y1)
    crop_x2 = min(img_w, x2)
    crop_y2 = min(img_h, crop_y2)

    cropped = image[crop_y1:crop_y2, crop_x1:crop_x2]
    crop_region = (crop_x1, crop_y1, crop_x2, crop_y2)

    return cropped, crop_region


def load_yolo_labels(label_path, img_w, img_h):
    """YOLO 라벨 파일을 읽어 픽셀 좌표로 변환"""
    labels = []
    if not os.path.exists(label_path):
        return labels

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

            # 정규화 좌표 → 픽셀 좌표
            px_cx = cx * img_w
            px_cy = cy * img_h
            px_w = bw * img_w
            px_h = bh * img_h
            lx1 = px_cx - px_w / 2
            ly1 = px_cy - px_h / 2
            lx2 = px_cx + px_w / 2
            ly2 = px_cy + px_h / 2

            labels.append({
                'class_id': cls_id,
                'bbox': (int(lx1), int(ly1), int(lx2), int(ly2)),
                'original_line': line.strip()
            })
    return labels


def calculate_contain_ratio(label_bbox, crop_region):
    """라벨 bbox가 크롭 영역에 몇 % 포함되는지 계산"""
    lx1, ly1, lx2, ly2 = label_bbox
    cx1, cy1, cx2, cy2 = crop_region

    inter_x1 = max(lx1, cx1)
    inter_y1 = max(ly1, cy1)
    inter_x2 = min(lx2, cx2)
    inter_y2 = min(ly2, cy2)

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    label_area = (lx2 - lx1) * (ly2 - ly1)

    return inter_area / label_area if label_area > 0 else 0


def match_labels_to_crop(labels, crop_region, crop_w, crop_h):
    """크롭 영역에 포함되는 라벨만 추출, 좌표를 크롭 기준으로 변환"""
    cx1, cy1, cx2, cy2 = crop_region
    matched = []

    for label in labels:
        ratio = calculate_contain_ratio(label['bbox'], crop_region)
        if ratio < LABEL_CONTAIN_RATIO:
            continue

        lx1, ly1, lx2, ly2 = label['bbox']

        # 크롭 기준 좌표로 변환 (클리핑)
        new_x1 = max(0, lx1 - cx1)
        new_y1 = max(0, ly1 - cy1)
        new_x2 = min(crop_w, lx2 - cx1)
        new_y2 = min(crop_h, ly2 - cy1)

        if new_x2 <= new_x1 or new_y2 <= new_y1:
            continue

        # YOLO 정규화 좌표로 변환
        ncx = ((new_x1 + new_x2) / 2) / crop_w
        ncy = ((new_y1 + new_y2) / 2) / crop_h
        nw = (new_x2 - new_x1) / crop_w
        nh = (new_y2 - new_y1) / crop_h

        matched.append({
            'class_id': label['class_id'],
            'yolo_line': f"{label['class_id']} {ncx:.6f} {ncy:.6f} {nw:.6f} {nh:.6f}",
            'pixel_bbox': (new_x1, new_y1, new_x2, new_y2)
        })

    return matched


def has_duplicate_class(matched_labels):
    """같은 클래스 라벨이 2개 이상이면 True (중복 필터링)"""
    class_counts = {}
    for m in matched_labels:
        cid = m['class_id']
        class_counts[cid] = class_counts.get(cid, 0) + 1
    for count in class_counts.values():
        if count >= 2:
            return True
    return False


def get_wear_status_code(matched_labels):
    """매칭된 라벨에서 착용 상태 코드 생성 (헬멧/조끼)"""
    class_ids = set(m['class_id'] for m in matched_labels)

    # 헬멧 상태: 0=착용, 2=미착용
    if 0 in class_ids:
        helmet = '1'
    elif 2 in class_ids:
        helmet = '0'
    else:
        helmet = '0'  # 라벨 없으면 기본 착용으로

    # 조끼 상태: 1=착용, 3=미착용
    if 1 in class_ids:
        vest = '1'
    elif 3 in class_ids:
        vest = '0'
    else:
        vest = '0'  # 라벨 없으면 기본 착용으로

    return helmet + vest

라벨 매칭 함수

미리 보기

In [ ]:
def preview_crops(split, num_images=1):
    """지정 split의 첫 번째 이미지로 크롭 결과 미리보기"""
    image_dir = os.path.join(BASE_DIR, "images", split)
    label_dir = os.path.join(BASE_DIR, "labels", split)

    image_files = sorted([f for f in os.listdir(image_dir)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    if not image_files:
        print(f"[{split}] 이미지 없음")
        return

    for img_idx in range(min(num_images, len(image_files))):
        fname = image_files[img_idx]
        img_path = os.path.join(image_dir, fname)
        label_path = os.path.join(label_dir, os.path.splitext(fname)[0] + '.txt')

        image = cv2.imread(img_path)
        if image is None:
            continue
        img_h, img_w = image.shape[:2]

        # 사람 감지
        results = model(image, conf=PERSON_CONFIDENCE, verbose=False)
        persons = []
        for box in results[0].boxes:
            if int(box.cls[0]) == 0:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                persons.append({'bbox': (x1, y1, x2, y2), 'confidence': float(box.conf[0])})
        persons = remove_duplicate_boxes(persons)

        # 원본 라벨 로드
        labels = load_yolo_labels(label_path, img_w, img_h)

        # 원본 이미지에 bbox 표시
        display_img = image.copy()
        for p in persons:
            x1, y1, x2, y2 = p['bbox']
            cv2.rectangle(display_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(display_img, f"person {p['confidence']:.2f}",
                        (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        for lb in labels:
            lx1, ly1, lx2, ly2 = lb['bbox']
            cv2.rectangle(display_img, (lx1, ly1), (lx2, ly2), (0, 0, 255), 2)
            cls_name = CLASS_NAMES.get(lb['class_id'], str(lb['class_id']))
            cv2.putText(display_img, cls_name,
                        (lx1, ly1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        # 서브플롯 구성: 원본 + 각 크롭
        n_crops = len(persons)
        fig, axes = plt.subplots(1, 1 + n_crops, figsize=(6 + 4 * n_crops, 6))
        if n_crops == 0:
            axes = [axes]

        axes[0].imshow(cv2.cvtColor(display_img, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"[{split}] {fname}\nperson(초록) / label(빨강)")
        axes[0].axis('off')

        for i, p in enumerate(persons):
            cropped, crop_region = crop_upper_body(image, p['bbox'])
            crop_h_px, crop_w_px = cropped.shape[:2]

            if crop_w_px < MIN_CROP_SIZE or crop_h_px < MIN_CROP_SIZE:
                continue

            matched = match_labels_to_crop(labels, crop_region, crop_w_px, crop_h_px)

            # 크롭 이미지에 매칭 라벨 표시
            crop_display = cropped.copy()
            label_texts = []
            for m in matched:
                mx1, my1, mx2, my2 = map(int, m['pixel_bbox'])
                cv2.rectangle(crop_display, (mx1, my1), (mx2, my2), (0, 255, 255), 2)
                cls_name = CLASS_NAMES.get(m['class_id'], str(m['class_id']))
                label_texts.append(cls_name)
                cv2.putText(crop_display, cls_name,
                            (mx1, my1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

            # 중복 체크 및 상태 코드
            dup = has_duplicate_class(matched)
            code = get_wear_status_code(matched)

            title = f"person{i+1} [{code}]\n"
            title += f"labels: {', '.join(label_texts) if label_texts else 'none'}"
            if dup:
                title += "\n⚠ 중복 → 제외"

            if i + 1 < len(axes):
                axes[i + 1].imshow(cv2.cvtColor(crop_display, cv2.COLOR_BGR2RGB))
                axes[i + 1].set_title(title, fontsize=9)
                axes[i + 1].axis('off')

        plt.tight_layout()
        plt.show()

        # 라벨 텍스트 출력
        print(f"\n{'='*50}")
        print(f"[{split}] {fname} — 라벨 내용:")
        if labels:
            for lb in labels:
                cls_name = CLASS_NAMES.get(lb['class_id'], str(lb['class_id']))
                print(f"  class={lb['class_id']}({cls_name})  bbox={lb['bbox']}")
        else:
            print("  라벨 파일 없음")
        print(f"{'='*50}\n")

메인 처리 함수

In [ ]:
def process_all_splits():
    """train/val/test 전체 split을 순회하며 크롭 처리 (이어서 하기 지원)"""

    total_crops = 0
    total_skipped_dup = 0
    total_skipped_small = 0
    total_no_label = 0
    total_skipped_exist = 0
    split_stats = {}

    # ★ 이미 처리된 파일 목록 로드 (이어서 하기용)
    existing_files = set()
    out_img_dir = os.path.join(CROP_OUTPUT_DIR, "images")
    if os.path.exists(out_img_dir):
        existing_files = set(os.listdir(out_img_dir))
    print(f"기존 크롭 파일: {len(existing_files)}개 (이미 처리된 원본은 건너뜁니다)\n")

    for split in SPLITS:
        image_dir = os.path.join(BASE_DIR, "images", split)
        label_dir = os.path.join(BASE_DIR, "labels", split)

        if not os.path.exists(image_dir):
            print(f"[{split}] 이미지 폴더 없음: {image_dir}")
            continue

        image_files = sorted([f for f in os.listdir(image_dir)
                              if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

        print(f"[{split}] 이미지 {len(image_files)}장 처리 시작...")

        split_crops = 0
        split_dup = 0
        split_small = 0
        split_no_label = 0
        split_skipped = 0

        # ★ PPE 착용 상태 카운터 — 기존 크롭 파일에서 초기값 로드
        helmet_on = 0
        helmet_off = 0
        vest_on = 0
        vest_off = 0

        for ef in existing_files:
            if ef.startswith(f"{split}_") and ef.endswith(('.jpg', '.png')):
                code = ef.rsplit('.', 1)[0].split('_')[-1]
                if len(code) == 2 and code.isdigit():
                    if code[0] == '1':
                        helmet_on += 1
                    else:
                        helmet_off += 1
                    if code[1] == '1':
                        vest_on += 1
                    else:
                        vest_off += 1

        if helmet_on + helmet_off > 0:
            print(f"  [{split}] 기존 크롭에서 복원 → 헬멧O:{helmet_on} X:{helmet_off} / 조끼O:{vest_on} X:{vest_off}")

        for idx, fname in enumerate(image_files):
            base_name = os.path.splitext(fname)[0]

            # ★ 이미 처리된 원본이면 건너뛰기
            prefix = f"{split}_{base_name}_"
            if any(f.startswith(prefix) for f in existing_files):
                split_skipped += 1
                continue

            img_path = os.path.join(image_dir, fname)
            label_path = os.path.join(label_dir, base_name + '.txt')

            image = cv2.imread(img_path)
            if image is None:
                continue
            img_h, img_w = image.shape[:2]

            # 1. 사람 감지
            results = model(image, conf=PERSON_CONFIDENCE, verbose=False)
            persons = []
            for box in results[0].boxes:
                if int(box.cls[0]) == 0:
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    persons.append({'bbox': (x1, y1, x2, y2), 'confidence': float(box.conf[0])})
            persons = remove_duplicate_boxes(persons)

            if not persons:
                continue

            # 2. 원본 라벨 로드
            labels = load_yolo_labels(label_path, img_w, img_h)

            # 3. 각 사람 크롭 처리
            base_name = os.path.splitext(fname)[0]
            for i, p in enumerate(persons):
                cropped, crop_region = crop_upper_body(image, p['bbox'])
                crop_h_px, crop_w_px = cropped.shape[:2]

                # 최소 크기 필터
                if crop_w_px < MIN_CROP_SIZE or crop_h_px < MIN_CROP_SIZE:
                    split_small += 1
                    continue

                # 라벨 매칭
                matched = match_labels_to_crop(labels, crop_region, crop_w_px, crop_h_px)

                # 매칭 라벨 없으면 스킵
                if not matched:
                    split_no_label += 1
                    continue

                # 클래스 중복 필터링
                if has_duplicate_class(matched):
                    split_dup += 1
                    continue

                # 착용 상태 코드
                code = get_wear_status_code(matched)

                # ★ PPE 카운터 업데이트
                if code[0] == '1':
                    helmet_on += 1
                else:
                    helmet_off += 1
                if code[1] == '1':
                    vest_on += 1
                else:
                    vest_off += 1

                # 파일명: split_원본이름_person번호_상태코드
                out_name = f"{split}_{base_name}_person{i+1}_{code}"
                img_out = os.path.join(CROP_OUTPUT_DIR, "images", out_name + ".jpg")
                lbl_out = os.path.join(CROP_OUTPUT_DIR, "labels", out_name + ".txt")

                # 저장
                cv2.imwrite(img_out, cropped)
                with open(lbl_out, 'w') as f:
                    for m in matched:
                        f.write(m['yolo_line'] + '\n')

                split_crops += 1

            # 진행률 표시 (100장마다) + PPE 착용 상태 집계
            if (idx + 1) % 100 == 0:
                print(f"  [{split}] {idx + 1}/{len(image_files)} 처리됨 (크롭: {split_crops}) | "
                      f"헬멧O:{helmet_on} X:{helmet_off} / 조끼O:{vest_on} X:{vest_off}")

        # split 결과
        split_stats[split] = {
            'images': len(image_files),
            'crops': split_crops,
            'dup': split_dup,
            'small': split_small,
            'no_label': split_no_label,
            'skipped': split_skipped
        }
        total_crops += split_crops
        total_skipped_dup += split_dup
        total_skipped_small += split_small
        total_no_label += split_no_label
        total_skipped_exist += split_skipped

        print(f"  [{split}] 완료 — 크롭: {split_crops}, 건너뜀: {split_skipped}, "
              f"중복제외: {split_dup}, 크기미달: {split_small}, 라벨없음: {split_no_label}")
        print()

    # 최종 결과
    print("=" * 60)
    print("전체 처리 결과")
    print("=" * 60)
    for split, stats in split_stats.items():
        print(f"  [{split:5s}] 원본: {stats['images']:5d}장 → 크롭: {stats['crops']:5d}장")
    print(f"  {'─' * 50}")
    print(f"  총 크롭: {total_crops}장 (신규)")
    print(f"  건너뜀 (이미 처리됨): {total_skipped_exist}")
    print(f"  제외 — 중복: {total_skipped_dup}, 크기미달: {total_skipped_small}, 라벨없음: {total_no_label}")
    print(f"\n  저장 위치: {CROP_OUTPUT_DIR}")
    print(f"    ├── images/")
    print(f"    └── labels/")
    print("=" * 60)

    return split_stats

착용 상태 측정 함수

In [ ]:
def count_ppe_status():
    """크롭된 이미지의 파일명에서 착용 상태 집계"""
    img_dir = os.path.join(CROP_OUTPUT_DIR, "images")
    files = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]

    stats = {'helmet_on': 0, 'helmet_off': 0, 'vest_on': 0, 'vest_off': 0}
    code_counts = {'11': 0, '10': 0, '01': 0, '00': 0}

    for fname in files:
        code = fname.rsplit('.', 1)[0].split('_')[-1]
        if len(code) == 2 and code.isdigit():
            if code in code_counts:
                code_counts[code] += 1

            if code[0] == '1':
                stats['helmet_on'] += 1
            else:
                stats['helmet_off'] += 1

            if code[1] == '1':
                stats['vest_on'] += 1
            else:
                stats['vest_off'] += 1

    print("\n" + "=" * 40)
    print("PPE 착용 상태 집계")
    print("=" * 40)
    print(f"  전체 이미지:   {len(files)}장")
    print(f"  헬멧 착용(1_): {stats['helmet_on']}장")
    print(f"  헬멧 미착용(0_): {stats['helmet_off']}장")
    print(f"  조끼 착용(_1): {stats['vest_on']}장")
    print(f"  조끼 미착용(_0): {stats['vest_off']}장")
    print(f"\n  상태 코드별:")
    print(f"    11 (헬멧O 조끼O): {code_counts['11']}장")
    print(f"    10 (헬멧O 조끼X): {code_counts['10']}장")
    print(f"    01 (헬멧X 조끼O): {code_counts['01']}장")
    print(f"    00 (헬멧X 조끼X): {code_counts['00']}장")
    print("=" * 40)

    return stats, code_counts

실행

In [ ]:
# --- 1단계: 미리보기 (선택사항) ---
# train 폴더 첫 번째 이미지로 결과 확인
# preview_crops("train", num_images=1)

# --- 2단계: 전체 처리 ---
split_stats = process_all_splits()

# --- 3단계: 착용 상태 집계 ---
count_ppe_status()

기존 크롭 파일: 38101개 (이미 처리된 원본은 건너뜁니다)

[train] 이미지 17248장 처리 시작...
  [train] 기존 크롭에서 복원 → 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 1400/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 2200/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 3600/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 5000/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 7200/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 7800/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 8000/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 10200/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 14200/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 14400/17248 처리됨 (크롭: 0) | 헬멧O:22024 X:16077 / 조끼O:3162 X:34939
  [train] 16200/17248 처리됨 (크롭: 41) | 헬멧O:22031 X:16111 / 조끼O:3201 X:34941
  [train] 16300/17248 처리됨 (크롭: 126) | 헬멧O:22062 X:16165 / 조끼O:3283 X:34944
  [

({'helmet_on': 29406, 'helmet_off': 20928, 'vest_on': 4986, 'vest_off': 45348},
 {'11': 4082, '10': 25324, '01': 904, '00': 20024})